In [ ]:
import itertools
import logging
import os
import pickle
import random
import sys
from os.path import join

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

In [ ]:
df_test = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_test_with_ESM1b_ts.pkl")
)
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_train_with_ESM1b_ts.pkl")
)
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    ids = []
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)
        ids.append(ind)
        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y, ids)


train_X, train_y, _ = create_input_and_output_data(df=df_train)
test_X, test_y, _ = create_input_and_output_data(df=df_test)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant4.pkl")
)
p450plants = [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4]

train_Xloop = []
train_yloop = []
test_Xloop = []
test_yloop = []

train_idsloop = []
test_idsloop = []

train_set_loop = []
test_set_loop = []
num_folds = 5

for i in range(num_folds):
    train_cache = pd.concat(p450plants[:i] + p450plants[i + 1 :])
    print("i", i)
    print("i+1", i + 1)
    print(len(p450plants[:i]))
    print(len(p450plants[i + 1 :]))

    train_cache.reset_index(inplace=True, drop=True)

    Mou_X_train, Mou_y_train, train_ids = create_input_and_output_data(df=train_cache)
    Mou_X_test, Mou_y_test, test_ids = create_input_and_output_data(df=p450plants[i])

    train_Xloop.append(np.concatenate([train_X, Mou_X_train]))
    train_yloop.append(np.concatenate([train_y, Mou_y_train]))
    test_Xloop.append(Mou_X_test)
    test_yloop.append(Mou_y_test)

    train_idsloop.append(train_ids)
    test_idsloop.append(test_ids)

    train_set_loop.append(pd.concat([df_train] + p450plants[:i] + p450plants[i + 1 :]))
    test_set_loop.append(p450plants[i])

    print("--------------------")

In [ ]:
for i in range(num_folds):
    param = {
        "learning_rate": 0.31553117247348733,
        "max_delta_step": 1.7726044219753656,
        "max_depth": 10,
        "min_child_weight": 1.3845040588450772,
        "num_rounds": 342.68325188584106,
        "reg_alpha": 0.531395259755843,
        "reg_lambda": 3.744980563764689,
        "weight": 0.26187490421514203,
    }

    num_round = param["num_rounds"]
    param["objective"] = "binary:logistic"
    param["eval_metric"] = ["error", "logloss"]

    weightss = np.array(
        [
            param["weight"] if binding == 0 else 1.0
            for binding in train_set_loop[i]["Binding"]
        ]
    )

    del param["num_rounds"]
    del param["weight"]

    dtrain = xgb.DMatrix(
        np.array(train_Xloop[i]),
        weight=weightss,
        label=np.array(train_yloop[i]),
        feature_names=feature_names,
    )
    dtest = xgb.DMatrix(
        np.array(test_X), label=np.array(test_y), feature_names=feature_names
    )
    dtest_new = xgb.DMatrix(
        np.array(test_Xloop[i]),
        label=np.array(test_yloop[i]),
        feature_names=feature_names,
    )

    evallist = [(dtest_new, "eval"), (dtrain, "train")]

    bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
    # -------------------------------------------------------------------------------------------
    y_test_pred = np.round(bst.predict(dtest))
    acc_test = np.mean(y_test_pred == np.array(test_y))
    roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

    index_of_ones = np.where(np.array(test_y) == 1)[0]
    values_of_ones = bst.predict(dtest)[index_of_ones]
    acc_1 = np.mean(np.round(values_of_ones) == 1)

    index_of_zeros = np.where(np.array(test_y) == 0)[0]
    values_of_zeros = bst.predict(dtest)[index_of_zeros]
    acc_0 = np.mean(np.round(values_of_zeros) == 0)
    print("------------------------%d----------------------------------" % i)
    print(
        "Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc)
    )
    print("Accuracy on 1 set: %s, Accuracy on 0 set: %s" % (acc_1, acc_0))
    # -------------------------------------------------------------------------------------------
    y_test_new_pred = np.round(bst.predict(dtest_new))
    acc_test_new = np.mean(y_test_new_pred == np.array(test_yloop[i]))
    try:
        roc_auc_new = roc_auc_score(np.array(test_yloop[i]), bst.predict(dtest_new))
    except:
        roc_auc_new = 0
    try:
        mcc = matthews_corrcoef(np.array(test_yloop[i]), y_test_new_pred)
    except:
        mcc = 0
    index_of_ones = np.where(np.array(test_yloop[i]) == 1)[0]
    values_of_ones = bst.predict(dtest_new)[index_of_ones]
    acc_1_new = np.mean(np.round(values_of_ones) == 1)

    index_of_zeros = np.where(np.array(test_yloop[i]) == 0)[0]
    values_of_zeros = bst.predict(dtest_new)[index_of_zeros]
    acc_0_new = np.mean(np.round(values_of_zeros) == 0)

    print(
        "Accuracy on test set: %s, ROC-AUC score for test set: %s, MCC: %s"
        % (acc_test_new, roc_auc_new, mcc)
    )
    print("Accuracy on 1 set: %s, Accuracy on 0 set: %s" % (acc_1_new, acc_0_new))
    print(len(train_set_loop[i]))
    print(len(test_set_loop[i]))
    print("-------------------------------------------------------")
    print("\n\n")
    # -------------------------------------------------------------------------------------------
    pickle.dump(bst, open(join(our_data + "slice" + str(i) + "model.dat"), "wb"))

    np.save(
        join(our_data + "train" + str(i) + "_y_test_pred.npy"), bst.predict(dtest_new)
    )
    np.save(join(our_data + "train" + str(i) + "_y_test.npy"), np.array(test_yloop[i]))

In [ ]:
p450plants[0]

In [ ]:
test_set_loop[0]["enzyme"]

In [ ]:
train_set_loop[0][train_set_loop[0]["Uniprot ID"].isnull()]["enzyme"]

In [ ]:
intersection = set(test_set_loop[0]["enzyme"]).intersection(
    set(train_set_loop[0][train_set_loop[0]["Uniprot ID"].isnull()]["enzyme"])
)

if len(intersection) > 0:
    common_values = list(intersection)
    print("same:", common_values)
else:
    print("no same")

In [ ]:
intersection = set(test_set_loop[2]["enzyme"]).intersection(
    set(train_set_loop[2][train_set_loop[2]["Uniprot ID"].isnull()]["enzyme"])
)

if len(intersection) > 0:
    common_values = list(intersection)
    print("same:", common_values)
else:
    print("no same")

In [ ]:
p450plants[0][p450plants[0]["enzyme"] == "CYP726A17"]

In [ ]:
p450plants[4][p450plants[4]["enzyme"] == "CYP726A17"]

In [ ]:
print(len(test_set_loop[0][test_set_loop[0]["Binding"] == 1]))
print(len(test_set_loop[0][test_set_loop[0]["Binding"] == 0]))

print(len(test_set_loop[1][test_set_loop[1]["Binding"] == 1]))
print(len(test_set_loop[1][test_set_loop[1]["Binding"] == 0]))

print(len(test_set_loop[2][test_set_loop[2]["Binding"] == 1]))
print(len(test_set_loop[2][test_set_loop[2]["Binding"] == 0]))

print(len(test_set_loop[3][test_set_loop[3]["Binding"] == 1]))
print(len(test_set_loop[3][test_set_loop[3]["Binding"] == 0]))

print(len(test_set_loop[4][test_set_loop[4]["Binding"] == 1]))
print(len(test_set_loop[4][test_set_loop[4]["Binding"] == 0]))